In [1]:
!pip install -q -U transformers langchain bitsandbytes qdrant-client
!pip install -q -U sentence-transformers accelerate unstructured

In [2]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 10.6 MB/s eta 0:00:00a 0:00:01


In [3]:
!pip install -q rank_bm25

In [8]:
from langchain.vectorstores import Qdrant
from langchain_community.document_loaders import DirectoryLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

file_path = "../supreme_court_texts/"

In [9]:
# Load legal case docs
loader = DirectoryLoader(file_path)
docs = loader.load()

In [10]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
split_docs = text_splitter.split_documents(docs)

In [11]:
# Use Legal-BERT for better embeddings
emb_model = "nlpaueb/legal-bert-base-uncased"
embeddings = HuggingFaceEmbeddings(model_name=emb_model)

/tmp/ipykernel_47749/1570553578.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=emb_model)
No sentence-transformers model found with name nlpaueb/legal-bert-base-uncased. Creating a new one with mean pooling.


In [12]:
qdrant_collection = Qdrant.from_documents(
docs,
embeddings,
location=":memory:", # Local mode with in-memory storage only
collection_name="legal_docs",
)

In [13]:
# construct a retriever on top of the vector store
qdrant_retriever = qdrant_collection.as_retriever()

In [14]:
# let's try a query and see the how its retrieved from the qdrant vector database
qdrant_retriever.invoke('Cite me a dispute related to electricity board tender')

[Document(metadata={'source': '../supreme_court_texts/316.txt', '_id': 'a0400a0b4cbf4b859b0370879e8fb4d8', '_collection_name': 'legal_docs'}, page_content='2024 INSC 328\n\n1\n\nREPORTABLE\n\nIN THE SUPREME COURT OF INDIA CIVIL APPELLATE JURISDICTION\n\nCIVIL APPEAL No.  4422/2024 (ARISING OUT OF SLP (C) No. 14475/2021\n\nYASH RAJ FILMS PRIVATE LIMITED\n\n…. APPELLANT(S) VERSUS AFREEN FATIMA ZAIDI & ANR.\n\n…RESPONDENT(S) J U D G M E N T\n\nPAMIDIGHANTAM SRI NARASIMHA, J. 1. What are the legal implications of a promotional trailer, popularly known as a ‘promo’, or a teaser that is circulated before the release of a movie? Does it create any contractual relationship or obligations akin to it? Is it an unfair trade practice if the contents of the promotional trailer are not shown in the movie? These questions have arisen in the context of a consumer dispute wherein the consumer courts have allowed the complaint alleging deficiency of service based on a ‘contractual obligation’ and ‘unfai